<a href="https://colab.research.google.com/github/shashwati-01/GenAI-Practical04-46-shashwati/blob/main/GenAi_Prac04_Shashwati_46.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementation of an LSTM-Based Encoder-Decoder Architecture for Sequence-to-Sequence Learning.

In [4]:
from pathlib import Path
import random
import string

import numpy as np
import tensorflow as tf
from tensorflow.keras import Model, layers
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 1. OPTIMIZED DATASET: 1,000 sequences is enough to learn without taking forever
corpus_path = Path("Int.txt")
if not corpus_path.is_file() or len(corpus_path.read_text(encoding="utf-8").splitlines()) < 1000:
    letters = string.ascii_lowercase
    sample_words = [
        "".join(random.choice(letters) for _ in range(random.randint(3, 8)))
        for _ in range(1000)
    ]
    corpus_path.write_text("\n".join(sample_words), encoding="utf-8")
    print("Created a sample input.txt with 1000 random sequences.")

lines = [line.strip().lower() for line in corpus_path.read_text(encoding="utf-8").splitlines()]
character_sequences = [line for line in lines if len(line) >= 3]

characters = sorted(set("".join(character_sequences)))
char_to_id = {character: index for index, character in enumerate(characters, start=1)}
id_to_char = {index: character for character, index in char_to_id.items()}

PAD_ID = 0
START_ID = len(char_to_id) + 1
END_ID = START_ID + 1
VOCAB_SIZE = END_ID + 1

max_input_length = max(map(len, character_sequences))
encoder_data = pad_sequences(
    [[char_to_id[character] for character in sequence] for sequence in character_sequences],
    maxlen=max_input_length,
    padding="post",
    dtype="int32",
)

target_sequences = [
    [START_ID]
    + [char_to_id[character] for character in sequence[::-1]]
    + [END_ID]
    for sequence in character_sequences
]
decoder_length = max_input_length + 1
decoder_input_data = pad_sequences(
    [sequence[:-1] for sequence in target_sequences],
    maxlen=decoder_length,
    padding="post",
    dtype="int32",
)
decoder_target_data = pad_sequences(
    [sequence[1:] for sequence in target_sequences],
    maxlen=decoder_length,
    padding="post",
    dtype="int32",
)
target_weights = (decoder_target_data != PAD_ID).astype("float32")

rng = np.random.default_rng(SEED)
indices = rng.permutation(len(encoder_data))
validation_size = max(1, int(0.2 * len(indices))) if len(indices) > 1 else 0
validation_indices = indices[:validation_size] if validation_size > 0 else indices
training_indices = indices[validation_size:] if validation_size > 0 else indices

# --- NEW CELL ---

# 2. SMALLER NETWORK: Drastically reduces parameter count for faster math
embedding_dim = 16  # Reduced to speed up computations
latent_dim = 64     # Reduced from 256. 64 is plenty for simple character tasks

encoder_tokens = layers.Input(shape=(max_input_length,), name="encoder_tokens")
token_embedding = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=embedding_dim,
    mask_zero=True,
    name="token_embedding",
)
encoder_lstm = layers.LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_embedded = token_embedding(encoder_tokens)
_, encoder_state_h, encoder_state_c = encoder_lstm(encoder_embedded)
encoder_states = [encoder_state_h, encoder_state_c]

decoder_tokens = layers.Input(shape=(decoder_length,), name="decoder_tokens")
decoder_lstm = layers.LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_embedded = token_embedding(decoder_tokens)
decoder_outputs, _, _ = decoder_lstm(decoder_embedded, initial_state=encoder_states)
token_classifier = layers.Dense(VOCAB_SIZE, activation="softmax", name="token_classifier")
decoder_probabilities = token_classifier(decoder_outputs)

model = Model([encoder_tokens, decoder_tokens], decoder_probabilities, name="lstm_seq2seq")
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["sparse_categorical_accuracy"],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
)

# 3. LARGER BATCH SIZE: Pushes more data through at once, minimizing overhead
history = model.fit(
    [encoder_data[training_indices], decoder_input_data[training_indices]],
    decoder_target_data[training_indices],
    sample_weight=target_weights[training_indices],
    validation_data=(
        [encoder_data[validation_indices], decoder_input_data[validation_indices]],
        decoder_target_data[validation_indices],
        target_weights[validation_indices],
    ),
    epochs=150,
    batch_size=256, # Increased from 64 to train much faster per epoch
    callbacks=[early_stopping],
    verbose=1,
)
print(f"Best validation loss: {min(history.history['val_loss']):.4f}")

# --- NEW CELL ---

encoder_model = Model(encoder_tokens, encoder_states, name="encoder_inference")

decoder_token_input = layers.Input(shape=(None,), dtype="int32", name="decoder_token_input")
state_h_input = layers.Input(shape=(latent_dim,), name="state_h_input")
state_c_input = layers.Input(shape=(latent_dim,), name="state_c_input")
decoder_embedding_input = token_embedding(decoder_token_input)
decoder_output, next_state_h, next_state_c = decoder_lstm(
    decoder_embedding_input,
    initial_state=[state_h_input, state_c_input],
)
step_probabilities = token_classifier(decoder_output)
decoder_model = Model(
    [decoder_token_input, state_h_input, state_c_input],
    [step_probabilities, next_state_h, next_state_c],
    name="decoder_inference",
)

def reverse_with_model(text):
    normalized_text = text.lower()
    encoded = pad_sequences(
        [[char_to_id.get(character, 0) for character in normalized_text if character in char_to_id]],
        maxlen=max_input_length,
        padding="post",
        dtype="int32",
    )
    state_h, state_c = encoder_model.predict(encoded, verbose=0)
    current_token = np.array([[START_ID]], dtype="int32")
    generated_characters = []

    for _ in range(len(normalized_text) + 1):
        probabilities, state_h, state_c = decoder_model.predict(
            [current_token, state_h, state_c],
            verbose=0,
        )
        predicted_id = int(np.argmax(probabilities[0, -1, :]))
        if predicted_id in (END_ID, PAD_ID, START_ID):
            break
        generated_characters.append(id_to_char[predicted_id])
        current_token = np.array([[predicted_id]], dtype="int32")

    return "".join(generated_characters)

# --- NEW CELL ---

test_words = ["hello", "world", "sequence", "learning", "model"]
for example in test_words:
    prediction = reverse_with_model(example)
    print(f"Input:     {example}")
    print(f"Expected:  {example[::-1]}")
    print(f"Predicted: {prediction}\n")

Epoch 1/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 192ms/step - loss: 3.3416 - sparse_categorical_accuracy: 0.1461 - val_loss: 3.2777 - val_sparse_categorical_accuracy: 0.1530
Epoch 2/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - loss: 3.1822 - sparse_categorical_accuracy: 0.1543 - val_loss: 3.1280 - val_sparse_categorical_accuracy: 0.1530
Epoch 3/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 3.1041 - sparse_categorical_accuracy: 0.1545 - val_loss: 3.0870 - val_sparse_categorical_accuracy: 0.1571
Epoch 4/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - loss: 3.0438 - sparse_categorical_accuracy: 0.1684 - val_loss: 2.9796 - val_sparse_categorical_accuracy: 0.1790
Epoch 5/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - loss: 2.9272 - sparse_categorical_accuracy: 0.1882 - val_loss: 2.8873 - val_sparse_categorical_accuracy: 0.1895
Epoch 6/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - loss: 2.8513 - sparse_categorical_accuracy: 0.2033 - val_loss: 2.8241 - val_sparse_categorical_accuracy: 0.2082
Epo